# GPT Fine-Tuning Code Explanation

This code demonstrates how to use pretrained GPT-2 for multiple NLP tasks:

- Entailment Classification
- Similarity Classification
- Multiple Choice QA
- Commonsense Reasoning

The pretrained GPT model is reused with different output heads.

---

# 1. Load GPT Model

```python
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt = AutoModel.from_pretrained("gpt2")
```

Purpose:
- load pretrained GPT-2
- load tokenizer
- convert text → token IDs

---

# 2. Tokenization

```python
def tok(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
```

Purpose:
- convert text into tensors
- automatically pad sequences
- create attention masks

Example:

```text
"A dog is running"
```

becomes:

```python
input_ids
attention_mask
```

---

# 3. Extract Final Hidden Representation

```python
def last_hidden(gpt, input_ids, attention_mask):
```

Purpose:
- run GPT forward pass
- get final token representation

---

## GPT Output

```python
out = gpt(
    input_ids=input_ids,
    attention_mask=attention_mask
)
```

GPT produces:

```python
last_hidden_state
```

Shape:

```text
(batch_size, seq_len, hidden_size)
```

Meaning:
- one vector per token

---

## Get Last Non-Padding Token

```python
last = attention_mask.sum(dim=1) - 1
```

Finds:
- final real token
- ignores padding

---

## Extract Final Representation

```python
hidden[torch.arange(hidden.size(0)), last]
```

Returns:

```text
final contextual sentence representation
```

This vector summarizes the input sequence.

---

# 4. GPTClassifier

```python
class GPTClassifier(nn.Module):
```

Used for:
- entailment
- sentiment
- topic classification
- text classification

---

## Forward Pass

```python
h = last_hidden(...)
```

Gets sentence representation.

Then:

```python
return self.fc(h)
```

Linear layer converts representation into:

```text
class logits
```

Example:

```text
[2.1, -1.0, 0.3]
```

Meaning:

| entailment | contradiction | neutral |
|---|---|---|
| 2.1 | -1.0 | 0.3 |

Highest value becomes prediction.

---

# 5. GPTSimilarity

```python
class GPTSimilarity(nn.Module):
```

Used for:
- semantic similarity
- paraphrase detection

---

## Two Inputs

```python
h1 = last_hidden(...)
h2 = last_hidden(...)
```

Each sentence gets separate GPT encoding.

---

## Merge Representations

```python
h1 + h2
```

Combines semantic meaning from both sentences.

Then classifier predicts:

```text
similar / not similar
```

---

# 6. GPTMultipleChoice

```python
class GPTMultipleChoice(nn.Module):
```

Used for:
- QA
- commonsense reasoning
- multiple-choice tasks

---

# Input Shape

```text
(batch_size, num_choices, seq_len)
```

Example:

```text
3 answer choices
```

---

# Flatten Choices

```python
input_ids.view(b * c, s)
```

Converts:

```text
(batch, choices, seq)
```

into:

```text
(batch*choices, seq)
```

so GPT can process all choices together.

---

# Score Each Choice

```python
scores = self.scorer(h)
```

Each choice gets:
- one score

Example:

| Choice | Score |
|---|---|
| green | -1.2 |
| blue | 3.5 |
| red | 0.2 |

Highest score = selected answer.

---

# 7. Training Functions

Three training loops:

| Function | Task |
|---|---|
| train_classifier | classification |
| train_similarity | similarity |
| train_multiple_choice | QA / commonsense |

---

# Training Process

For every example:

```python
logits = model(...)
```

Compute predictions.

---

## Compute Loss

```python
loss = loss_fn(logits, label)
```

Measures prediction error.

---

## Reset Old Gradients

```python
opt.zero_grad()
```

Clears previous gradients.

---

## Backpropagation

```python
loss.backward()
```

Computes gradients.

---

## Update GPT Parameters

```python
opt.step()
```

Updates:
- GPT weights
- classifier weights

This is:

```text
fine-tuning
```

---

# 8. Entailment Task

Input format:

```text
<start> premise <delim> hypothesis <extract>
```

Example:

```text
A dog is running.
An animal is outdoors.
```

Predict:
- entailment
- contradiction
- neutral

---

# 9. Similarity Task

Two sentence orders:

```python
text1 + text2
text2 + text1
```

Helps model learn:
- symmetric similarity

Predict:
- similar
- not similar

---

# 10. Question Answering

Input:

```text
context + question + answer choice
```

Example:

```text
The sky is blue.
What color is the sky?
blue
```

Each answer choice processed separately.

Model selects highest-scoring answer.

---

# 11. Commonsense Reasoning

Similar to QA.

Example:

```text
John put ice cream in the sun.
What happened next?
```

Choices:
- melted
- colder
- rock

Model learns logical continuation.

---

# 12. Prediction Phase

```python
model.eval()
```

Switches model to evaluation mode.

---

# Disable Gradients

```python
with torch.no_grad():
```

Purpose:
- save memory
- faster inference
- no training

---

# Final Prediction

```python
prediction = logits.argmax(dim=-1).item()
```

Chooses highest-scoring class.

---

# Overall Pipeline

```text
Input Text
    ↓
Tokenizer
    ↓
GPT-2
    ↓
Final Hidden Representation
    ↓
Task-Specific Head
    ↓
Prediction
```

---

# Key Idea

GPT acts as:

```text
general language encoder
```

Different heads adapt GPT to:
- classification
- similarity
- QA
- reasoning

using fine-tuning.

In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
gpt = AutoModel.from_pretrained("gpt2")


def last_hidden(gpt, input_ids, attention_mask):
    out = gpt(input_ids=input_ids, attention_mask=attention_mask)
    hidden = out.last_hidden_state
    last = attention_mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.size(0)), last]


class GPTClassifier(nn.Module):
    def __init__(self, gpt, num_labels):
        super().__init__()
        self.gpt = gpt
        self.fc = nn.Linear(gpt.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        h = last_hidden(self.gpt, input_ids, attention_mask)
        return self.fc(h)


class GPTSimilarity(nn.Module):
    def __init__(self, gpt):
        super().__init__()
        self.gpt = gpt
        self.fc = nn.Linear(gpt.config.hidden_size, 2)

    def forward(self, ids1, mask1, ids2, mask2):
        h1 = last_hidden(self.gpt, ids1, mask1)
        h2 = last_hidden(self.gpt, ids2, mask2)
        return self.fc(h1 + h2)


class GPTMultipleChoice(nn.Module):
    def __init__(self, gpt):
        super().__init__()
        self.gpt = gpt
        self.scorer = nn.Linear(gpt.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        b, c, s = input_ids.shape
        input_ids = input_ids.view(b * c, s)
        attention_mask = attention_mask.view(b * c, s)

        h = last_hidden(self.gpt, input_ids, attention_mask)
        scores = self.scorer(h)

        return scores.view(b, c)


def tok(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors="pt")


def train_classifier(model, data, formatter, epochs=10):
    opt = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total = 0

        for ex in data:
            text = formatter(ex)
            batch = tok([text])
            label = torch.tensor([ex["label"]])

            logits = model(batch["input_ids"], batch["attention_mask"])
            loss = loss_fn(logits, label)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total += loss.item()

        print(f"Epoch {epoch+1}, Loss={total:.4f}")


def train_similarity(model, data, formatter, epochs=10):
    opt = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total = 0

        for ex in data:
            t1, t2 = formatter(ex)
            b1 = tok([t1])
            b2 = tok([t2])
            label = torch.tensor([ex["label"]])

            logits = model(
                b1["input_ids"], b1["attention_mask"],
                b2["input_ids"], b2["attention_mask"]
            )

            loss = loss_fn(logits, label)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total += loss.item()

        print(f"Epoch {epoch+1}, Loss={total:.4f}")


def train_multiple_choice(model, data, formatter, epochs=10):
    opt = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total = 0

        for ex in data:
            choices = formatter(ex)
            batch = tok(choices)

            input_ids = batch["input_ids"].unsqueeze(0)
            mask = batch["attention_mask"].unsqueeze(0)
            label = torch.tensor([ex["label"]])

            scores = model(input_ids, mask)
            loss = loss_fn(scores, label)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total += loss.item()

        print(f"Epoch {epoch+1}, Loss={total:.4f}")

D:\anaconda3\envs\web_lumplt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------- Entailment ----------------

entailment_data = [
    {"premise": "A dog is running in the park.", "hypothesis": "An animal is outdoors.", "label": 0},
    {"premise": "The man is sleeping.", "hypothesis": "The man is running.", "label": 1},
    {"premise": "A woman is reading a book.", "hypothesis": "The woman is a teacher.", "label": 2},
]

entailment_labels = {0: "entailment", 1: "contradiction", 2: "neutral"}

def format_entailment(ex):
    return f"<start> {ex['premise']} <delim> {ex['hypothesis']} <extract>"

entail_model = GPTClassifier(gpt, num_labels=3)
train_classifier(entail_model, entailment_data, format_entailment)

Epoch 1, Loss=21.7646
Epoch 2, Loss=10.3176
Epoch 3, Loss=4.1724
Epoch 4, Loss=18.1308
Epoch 5, Loss=2.0586
Epoch 6, Loss=3.1532
Epoch 7, Loss=4.0658
Epoch 8, Loss=4.0468
Epoch 9, Loss=3.4548
Epoch 10, Loss=2.7100


In [3]:
# ---------------- Similarity ----------------

similarity_data = [
    {"text1": "A man is playing guitar.", "text2": "A person is making music.", "label": 1},
    {"text1": "A cat is sleeping.", "text2": "A car is driving.", "label": 0},
]

def format_similarity(ex):
    t1 = f"<start> {ex['text1']} <delim> {ex['text2']} <extract>"
    t2 = f"<start> {ex['text2']} <delim> {ex['text1']} <extract>"
    return t1, t2

sim_model = GPTSimilarity(gpt)
train_similarity(sim_model, similarity_data, format_similarity)

Epoch 1, Loss=6.0853
Epoch 2, Loss=2.3316
Epoch 3, Loss=0.5294
Epoch 4, Loss=1.4399
Epoch 5, Loss=0.7839
Epoch 6, Loss=0.3576
Epoch 7, Loss=0.4070
Epoch 8, Loss=0.2274
Epoch 9, Loss=0.0700
Epoch 10, Loss=0.0316


In [4]:
# ---------------- QA ----------------

qa_data = [
    {
        "context": "The sky is clear and blue.",
        "question": "What color is the sky?",
        "answers": ["green", "blue", "red"],
        "label": 1,
    }
]

def format_qa(ex):
    return [
        f"<start> {ex['context']} {ex['question']} <delim> {a} <extract>"
        for a in ex["answers"]
    ]

qa_model = GPTMultipleChoice(gpt)
train_multiple_choice(qa_model, qa_data, format_qa)

Epoch 1, Loss=1.3616
Epoch 2, Loss=1.1108
Epoch 3, Loss=1.0243
Epoch 4, Loss=0.9567
Epoch 5, Loss=0.8707
Epoch 6, Loss=0.7321
Epoch 7, Loss=0.5082
Epoch 8, Loss=0.2981
Epoch 9, Loss=0.1655
Epoch 10, Loss=0.0496


In [5]:
# ---------------- Commonsense ----------------

commonsense_data = [
    {
        "context": "John put ice cream in the sun.",
        "question": "What happened next?",
        "answers": ["It melted.", "It became colder.", "It turned into a rock."],
        "label": 0,
    }
]

def format_commonsense(ex):
    return [
        f"<start> {ex['context']} {ex['question']} <delim> {a} <extract>"
        for a in ex["answers"]
    ]

common_model = GPTMultipleChoice(gpt)
train_multiple_choice(common_model, commonsense_data, format_commonsense)

Epoch 1, Loss=1.2779
Epoch 2, Loss=0.6872
Epoch 3, Loss=0.2886
Epoch 4, Loss=0.0761
Epoch 5, Loss=0.0148
Epoch 6, Loss=0.0029
Epoch 7, Loss=0.0006
Epoch 8, Loss=0.0002
Epoch 9, Loss=0.0001
Epoch 10, Loss=0.0000


In [7]:
# =====================================================
# ENTAILMENT PREDICTION
# =====================================================

entail_model.eval()

text = format_entailment(entailment_data[0])
print(text)
batch = tok([text])

with torch.no_grad():

    logits = entail_model(
        batch["input_ids"],
        batch["attention_mask"]
    )

prediction = logits.argmax(dim=-1).item()

print("Entailment Prediction:")
print(entailment_labels[prediction])


<start> A dog is running in the park. <delim> An animal is outdoors. <extract>
Entailment Prediction:
contradiction


In [8]:

# =====================================================
# SIMILARITY PREDICTION
# =====================================================

sim_model.eval()

t1, t2 = format_similarity(similarity_data[0])
print(similarity_data[0])
b1 = tok([t1])
b2 = tok([t2])

with torch.no_grad():

    logits = sim_model(
        b1["input_ids"],
        b1["attention_mask"],
        b2["input_ids"],
        b2["attention_mask"]
    )

prediction = logits.argmax(dim=-1).item()

print("\nSimilarity Prediction:")
print("similar" if prediction == 1 else "not similar")


{'text1': 'A man is playing guitar.', 'text2': 'A person is making music.', 'label': 1}

Similarity Prediction:
not similar


In [9]:
# =====================================================
# QUESTION ANSWERING PREDICTION
# =====================================================

qa_model.eval()

choices = format_qa(qa_data[0])
print(choices)
batch = tok(choices)

input_ids = batch["input_ids"].unsqueeze(0)
mask = batch["attention_mask"].unsqueeze(0)

with torch.no_grad():

    scores = qa_model(input_ids, mask)

prediction = scores.argmax(dim=-1).item()

print("\nQA Prediction:")
print(qa_data[0]["answers"][prediction])



['<start> The sky is clear and blue. What color is the sky? <delim> green <extract>', '<start> The sky is clear and blue. What color is the sky? <delim> blue <extract>', '<start> The sky is clear and blue. What color is the sky? <delim> red <extract>']

QA Prediction:
blue


In [10]:




# =====================================================
# COMMONSENSE PREDICTION
# =====================================================

common_model.eval()

choices = format_commonsense(commonsense_data[0])
print(choices)
batch = tok(choices)

input_ids = batch["input_ids"].unsqueeze(0)
mask = batch["attention_mask"].unsqueeze(0)

with torch.no_grad():

    scores = common_model(input_ids, mask)

prediction = scores.argmax(dim=-1).item()

print("\nCommonsense Prediction:")
print(commonsense_data[0]["answers"][prediction])

['<start> John put ice cream in the sun. What happened next? <delim> It melted. <extract>', '<start> John put ice cream in the sun. What happened next? <delim> It became colder. <extract>', '<start> John put ice cream in the sun. What happened next? <delim> It turned into a rock. <extract>']

Commonsense Prediction:
It melted.
